# Camera-calibration validation (Phase 1)
Runs the drift-free calibration on existing labeler clicks (clip + training) and
checks: FOLD-FREE (shallow frames don't project the whole field in), DRIFT-FREE
(held-out error flat across the full game), ACCURACY (held-out feet vs the free
homography). Local; no GPU.

In [ ]:
import json, numpy as np, cv2
from pathlib import Path
from collections import defaultdict
from soccer_vision.calib.calibrate import calibrate_camera
from soccer_vision.calib.validate import fold_count, leave_one_out_feet
from soccer_vision.pitch.landmarks import PITCH_LANDMARKS

CACHE = Path.home()/"sv-labeler/.sv_labeler_cache"
SESSIONS = {"clip": "clip.clicks.json", "training": "training.clicks.json"}
FRAME = (1920, 1080)

def load_obs(name):
    raw = json.loads((CACHE/SESSIONS[name]).read_text())
    clicks = raw if isinstance(raw, list) else raw["clicks"]
    obs = defaultdict(list)
    for c in clicks:
        obs[c["frame"]].append((c["kp_idx"], c["x"]*FRAME[0], c["y"]*FRAME[1]))
    return dict(obs)

In [ ]:
for name in SESSIONS:
    obs = load_obs(name)
    res = calibrate_camera(obs, FRAME, min_points=6)
    print(f"=== {name}: {len(res.frames)} calibrated frames, {res.n_excluded} excluded ===")
    print(f"  shared focal: {res.K[0,0]:.0f}px  (frame width {FRAME[0]})  reproj RMS median {np.median(list(res.rms_px.values())):.1f}px")
    # FOLD-FREE: landmarks in-frame per calibrated frame (folding=~21, slice=~6-12)
    folds = [fold_count(res.pitch_homography(f), FRAME) for f in res.frames]
    print(f"  fold check: in-frame landmarks/frame min {min(folds)} median {int(np.median(folds))} max {max(folds)}  (folding would be ~21)")
    # ACCURACY: held-out feet
    errs = leave_one_out_feet({f: obs[f] for f in res.frames}, res.K, FRAME, min_other=4)
    allv = [v for vals in errs.values() for v in vals]
    print(f"  held-out accuracy: median {np.median(allv):.1f}ft  90th {np.percentile(allv,90):.1f}ft")
    # DRIFT-FREE: per-frame fit RMS early-vs-late. The camera calib solves each
    # frame independently against the field (no chaining), so RMS should be FLAT
    # across the whole game (contrast the chained homography's 4->95 ft growth).
    fr = np.array(sorted(res.frames)); rms_arr = np.array([res.rms_px[f] for f in fr])
    third = max(1, len(fr) // 3)
    print(f"  drift check (per-frame fit RMS): early-third {np.median(rms_arr[:third]):.1f}px "
          f"vs late-third {np.median(rms_arr[-third:]):.1f}px  (flat = drift-free)")

## Overlays (Patrick assesses)
The next cell renders the calibrated pitch on sampled frames for visual review.
Claude renders; Patrick interprets.

In [ ]:
EDGES=[(0,1),(1,3),(3,2),(2,0),(4,6),(9,10),(11,12),(9,11),(10,12),(13,14),(15,16),(13,15),(14,16),(17,18),(19,20)]
name="training"; obs=load_obs(name); res=calibrate_camera(obs,FRAME,min_points=6)
cap=cv2.VideoCapture(str(Path.home()/f"sv-labeler/{name}.mp4")); cells=[]
import random; random.seed(0)
for f in sorted(random.sample(res.frames, min(8,len(res.frames)))):
    cap.set(cv2.CAP_PROP_POS_FRAMES,f); ok,img=cap.read()
    if not ok: continue
    Hp=res.pitch_homography(f); canon=np.column_stack([PITCH_LANDMARKS[:,0],PITCH_LANDMARKS[:,1],np.ones(21)])
    pr=(Hp@canon.T).T; uv=pr[:,:2]/pr[:,2:3]
    for a,b in EDGES:
        cv2.line(img,(int(uv[a,0]),int(uv[a,1])),(int(uv[b,0]),int(uv[b,1])),(40,200,255),3)
    cells.append(img)
cap.release()
cw=640; tiles=[cv2.resize(c,(cw,int(c.shape[0]*cw/c.shape[1]))) for c in cells]
rows=[np.hstack(tiles[i:i+4]+[np.zeros_like(tiles[0])]*(4-len(tiles[i:i+4]))) for i in range(0,len(tiles),4)]
cv2.imwrite("/tmp/calib_overlay.jpg",np.vstack(rows)); print("wrote /tmp/calib_overlay.jpg")